# Урок 7.3. Сложность деревьев и штрафы на листья

**Интерактивная версия:** `lessons/lesson_7_3/web/index.html`

1. λ: сжатие весов листьев — формула против дерева gbcourse.
2. Сетка λ × γ на шумных данных (среднее по разбиениям).
3. min_child_weight в классификации.
4. Те же параметры в XGBoost.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "python" / "gbcourse").is_dir())
sys.path.insert(0, str(ROOT / "shared" / "python"))

import numpy as np
import pandas as pd
from gbcourse import datasets, GBRegressor, GBClassifier, RegressionTree
from gbcourse.plotting import use_course_style

use_course_style()

## 1. λ сжимает листья: w = Σr / (n + λ)

In [2]:
X, y = datasets.regression_1d(kind="wave", n=60, noise=0.5, seed=1)
r = y - y.mean()
for lam in (0, 1, 10):
    t = RegressionTree(max_depth=2, reg_lambda=lam).fit(X, -r)
    leaf = t.leaves[0]
    print(f"λ = {lam:2d}: лист из {leaf.n} объектов, Σr = {-leaf.G:+.3f}, вес {leaf.value:+.4f}, формула {-leaf.G / (leaf.n + lam):+.4f}")

λ =  0: лист из 4 объектов, Σr = -5.136, вес -1.2840, формула -1.2840
λ =  1: лист из 21 объектов, Σr = -9.244, вес -0.4202, формула -0.4202
λ = 10: лист из 35 объектов, Σr = -23.999, вес -0.5333, формула -0.5333


## 2. Сетка λ × γ

In [3]:
Xf, yf = datasets.friedman1(n=300, noise=2.0, seed=75)


def score(**kw):
    out = []
    for split in range(3):
        a, b, c, d = datasets.train_test_split(Xf, yf, test_size=0.3, seed=split)
        out.append(min(GBRegressor(mode="newton", n_estimators=200, learning_rate=0.1, max_depth=5, **kw).fit(a, c, eval_set=(b, d)).history_["eval"]))
    return np.mean(out)


grid = pd.DataFrame({f"γ = {g}": [score(reg_lambda=lam, gamma=g) for lam in (0, 5, 20)] for g in (0, 10, 40)},
                    index=[f"λ = {lam}" for lam in (0, 5, 20)])
grid.round(3)

,γ = 0,γ = 10,γ = 40
λ = 0,3.861,4.335,4.870
λ = 5,3.724,4.089,4.797
λ = 20,3.738,4.170,5.252


## 3. min_child_weight в классификации

Для log-loss гессиан $p(1-p)$ мал у уверенных объектов — «вес» листа меньше числа объектов.

In [4]:
Xc, yc = datasets.classification_2d(kind="moons", n=400, noise=0.3, seed=78)
a, b, c, d = datasets.train_test_split(Xc, yc, test_size=0.3, seed=0)
for mcw in (0, 1, 5):
    m = GBClassifier(mode="newton", n_estimators=300, learning_rate=0.3, max_depth=4, min_child_weight=mcw).fit(a, c, eval_set=(b, d))
    leaves = np.mean([t.n_leaves for (t,) in m.trees_])
    print(f"min_child_weight = {mcw}: листьев в среднем {leaves:.1f}, лучший log-loss {min(m.history_['eval']):.4f}")

min_child_weight = 0: листьев в среднем 13.7, лучший log-loss 0.2693


min_child_weight = 1: листьев в среднем 4.6, лучший log-loss 0.2716
min_child_weight = 5: листьев в среднем 2.9, лучший log-loss 0.2873


## 4. XGBoost

In [5]:
import xgboost as xgb

a, b, c, d = datasets.train_test_split(Xf, yf, test_size=0.3, seed=0)
for kw in (dict(reg_lambda=0), dict(reg_lambda=5), dict(reg_lambda=5, gamma=10), dict(reg_lambda=5, min_child_weight=10)):
    m = xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, **kw).fit(a, c)
    print(f"{str(kw):45s} MSE валидации {np.mean((d - m.predict(b)) ** 2):.3f}")

{'reg_lambda': 0}                             MSE валидации 6.986
{'reg_lambda': 5}                             MSE валидации 6.804
{'reg_lambda': 5, 'gamma': 10}                MSE валидации 7.422
{'reg_lambda': 5, 'min_child_weight': 10}     MSE валидации 6.185


## Упражнения

1. Выведите вес листа с L1-штрафом α для L2-потерь и проверьте, когда он равен нулю.
2. Постройте зависимость среднего числа листьев от γ (0…50) на задаче Фридмана.

Решения: `python lessons/lesson_7_3/exercises/solutions.py`.